### Client API, returns raw json

In [1]:
import requests, json
import pandas as pd
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)

url  = 'https://api.deadlock-api.com/v1/matches/metadata'


params = {
    "limit": 500,
    "order_by": "start_time",
    "order_direction": "desc",
    "match_mode": "ranked",
    "include_info": "true",
    "include_more_info": "true",
    "include_objectives": "true",
    "include_mid_boss": "true",
    "include_player_info": "true",
    "include_player_final_stats": "true",
    "include_player_stats": "true",
    "include_player_items": "true",
    "include_player_death_details": "true",
    "hero_ids": "77", # Only get hero ids for apollo
    "format": "json",
}
response = requests.get(url, params=params, timeout=30)
response.raise_for_status()
matches = response.json()

# print(len(matches))
# print(json.dumps(matches, indent=2)[:3000])
matches_df = pd.DataFrame(matches)
matches_df.head()

,match_id,start_time,winning_team,duration_s,match_outcome,match_mode,game_mode,average_badge_team0,average_badge_team1,average_badge,not_scored,rewards_eligible,is_high_skill_range_parties,low_pri_pool,new_player_pool,team_score,match_tracked_stats,team0_tracked_stats,team1_tracked_stats,ranked_type,rank_interval,mid_boss,objectives,players,banned_hero_ids
0,102780404,2026-08-31 03:33:10,Team1,1500,TeamWin,Ranked,Normal,0,0,51,False,True,False,False,False,[],{},{},{},Normal,1,"[{'team_killed': 'Team1', 'team_claimed': 'Tea...","[{'destroyed_time_s': 648, 'creep_damage': 214...","[{'account_id': 1720100917, 'hero_id': 12, 'pl...",[]
1,102779854,2026-08-31 03:29:52,Team0,2013,TeamWin,Ranked,Normal,0,0,85,False,True,False,False,False,[],{},{},{},Normal,1,"[{'team_killed': 'Team1', 'team_claimed': 'Tea...","[{'destroyed_time_s': 871, 'creep_damage': 192...","[{'account_id': 10397429, 'hero_id': 77, 'play...",[]
2,102779825,2026-08-31 03:29:33,Team0,1705,TeamWin,Ranked,Normal,0,0,41,False,True,False,False,False,[],{},{},{},Normal,1,"[{'team_killed': 'Team0', 'team_claimed': 'Tea...","[{'destroyed_time_s': 737, 'creep_damage': 148...","[{'account_id': 186468851, 'hero_id': 81, 'pla...",[]
3,102778788,2026-08-31 03:23:06,Team1,1805,TeamWin,Ranked,Normal,0,0,91,False,True,False,False,False,[],{},{},{},Normal,1,"[{'team_killed': 'Team1', 'team_claimed': 'Tea...","[{'destroyed_time_s': 451, 'creep_damage': 194...","[{'account_id': 1105767319, 'hero_id': 63, 'pl...",[]
4,102778717,2026-08-31 03:22:38,Team0,2146,TeamWin,Ranked,Normal,0,0,25,False,True,False,False,False,[],{},{},{},Normal,1,"[{'team_killed': 'Team0', 'team_claimed': 'Tea...","[{'destroyed_time_s': 868, 'creep_damage': 130...","[{'account_id': 1548210166, 'hero_id': 64, 'pl...",[]


In [2]:
matches_df.info()
matches_df["match_outcome"].value_counts()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 25 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   match_id                     500 non-null    int64 
 1   start_time                   500 non-null    str   
 2   winning_team                 500 non-null    str   
 3   duration_s                   500 non-null    int64 
 4   match_outcome                500 non-null    str   
 5   match_mode                   500 non-null    str   
 6   game_mode                    500 non-null    str   
 7   average_badge_team0          500 non-null    int64 
 8   average_badge_team1          500 non-null    int64 
 9   average_badge                500 non-null    int64 
 10  not_scored                   500 non-null    bool  
 11  rewards_eligible             500 non-null    bool  
 12  is_high_skill_range_parties  500 non-null    bool  
 13  low_pri_pool                 500 non-null    b

match_outcome
TeamWin    500
Name: count, dtype: int64

In [3]:
# Explode the lists into individual rows
matches_objectives = matches_df[["match_id", "objectives"]].explode("objectives")

# Normalize the dictionaries AND preserve the original index alignment
normalized_df = pd.json_normalize(matches_objectives['objectives'])
normalized_df.index = matches_objectives.index

# Join them safely without mismatched rows
matches_objectives = matches_objectives[["match_id"]].join(normalized_df)
matches_objectives.head()


,match_id,destroyed_time_s,creep_damage,creep_damage_mitigated,player_damage,player_damage_mitigated,first_damage_time_s,team_objective,team,player_spirit_damage
0,102780404,648,2148,0,3375,0,48,Tier1Lane3,Team0,1618
0,102780404,0,3970,0,1279,0,51,Tier1Lane4,Team1,217
0,102780404,626,2921,0,2591,0,174,Tier1Lane1,Team0,779
0,102780404,718,2579,0,2928,0,114,Tier1Lane3,Team1,1545
0,102780404,945,2597,0,2954,0,136,Tier1Lane1,Team1,1893


### Filters, players from each match, _includes every player currently_

In [4]:
player_rows = []
for match in matches:
    for p in match["players"]:
        p["match_id"] = match["match_id"]
        player_rows.append(p)

players_df = pd.DataFrame(player_rows)
players_df.head(1)

,account_id,hero_id,player_slot,team,hero_build_id,pregame_hero_id,kills,deaths,assists,net_worth,last_hits,denies,ability_points,assigned_lane,player_level,abandon_match_time_s,mvp_rank,player_tracked_stats,accolades,hero_xp_rewards,player_match_outcome,player_rank_initial_display_rank,player_rank_initial_flat_progress,player_rank_final_flat_progress,player_rank_desired_progress_change,player_rank_initial_calibration_games,player_rank_initial_demotion_protection_games,player_rank_consumed_demotion_protection,player_rank_initial_win_streak,items,stats,final_stats,death_details,match_id
0,1720100917,12,7,Team1,0,0,8,0,22,29526,97,0,25,6,29,0,2.0,{},"[{'accolade_id': 18, 'accolade_stat_value': 1,...","[{'hero_id': 12, 'xp_grant': 100, 'reason': 'W...",Win,55,32700,33000,300,0,2,False,0,"[{'game_time_s': 46, 'item_id': 18921423, 'upg...","[{'time_stamp_s': 180, 'net_worth': 1681, 'gol...","{'ability_kills': 7, 'ability_points': 25, 'ab...",[],102780404


In [5]:
# # Explode the lists into individual rows
# matches_objectives = matches[["match_id", "objectives"]].explode("objectives")
#
# # Normalize the dictionaries AND preserve the original index alignment
# normalized_df = pd.json_normalize(matches_objectives['objectives'])
# normalized_df.index = matches_objectives.index
#
# # Join them safely without mismatched rows
# matches_objectives = matches_objectives[["match_id"]].join(normalized_df)
# matches_objectives.head()


In [6]:
stats_df = players_df[['match_id', 'account_id', 'hero_id', 'stats']].explode('stats')
stats_df = stats_df.join(pd.json_normalize(stats_df['stats']))

In [7]:
stats_df.head(5)

,match_id,account_id,hero_id,stats,time_stamp_s,net_worth,gold_player,gold_player_orbs,gold_lane_creep_orbs,gold_neutral_creep_orbs,gold_boss,gold_boss_orb,gold_treasure,gold_denied,gold_death_loss,gold_lane_creep,gold_neutral_creep,kills,deaths,assists,creep_kills,neutral_kills,possible_creeps,creep_damage,player_damage,neutral_damage,boss_damage,denies,player_healing,ability_points,self_healing,player_barriering,teammate_healing,teammate_barriering,self_damage,bullet_kills,melee_kills,ability_kills,headshot_kills,player_damage_taken,max_health,weapon_power,tech_power,shots_hit,shots_missed,damage_absorbed,absorption_provided,hero_bullets_hit,hero_bullets_hit_crit,heal_prevented,heal_lost,damage_mitigated,level
0,102780404,1720100917,12,"{'time_stamp_s': 180, 'net_worth': 1681, 'gold...",180,1681,2,0,367,0,0,0,0,76,0,533,103,0,0,0,11,3,11,2808,932,298,0,0,716,2,305,0,411,0,0,0,0,0,0,1036,1060,0,20,105,45,0,0,13,5,0,0,208,4
0,102780404,1720100917,12,"{'time_stamp_s': 180, 'net_worth': 1681, 'gold...",360,3550,40,0,1117,0,0,0,0,129,0,1401,166,0,0,0,24,4,25,6339,1692,700,0,0,1871,4,936,0,935,0,0,0,0,0,0,2109,1403,0,24,205,105,0,0,24,9,0,0,762,7
0,102780404,1720100917,12,"{'time_stamp_s': 180, 'net_worth': 1681, 'gold...",540,5689,295,0,1826,0,0,0,0,129,0,2141,505,0,0,1,30,5,31,8156,3348,1250,0,0,3929,6,1935,0,1994,0,0,0,0,0,0,3699,1620,0,48,267,147,0,0,36,10,0,0,1052,10
0,102780404,1720100917,12,"{'time_stamp_s': 180, 'net_worth': 1681, 'gold...",720,9648,2146,0,2342,0,290,0,247,187,0,2930,505,3,0,2,36,5,37,10958,4789,1250,292,0,7339,11,3561,0,3778,0,0,1,0,2,0,5937,1971,0,80,326,239,0,0,68,14,0,0,1515,15
0,102780404,1720100917,12,"{'time_stamp_s': 180, 'net_worth': 1681, 'gold...",900,13580,3203,0,2680,0,622,0,247,187,0,3335,2179,4,0,4,38,15,39,12670,8005,4621,1590,0,10134,16,5071,0,5063,0,0,1,0,3,0,7216,2543,0,126,358,255,0,0,82,21,0,233,1874,20


In [8]:
items_df = players_df[['match_id', 'account_id', 'hero_id', 'items']].explode('items')
items_df = items_df.join(pd.json_normalize(items_df['items']))

In [9]:
url = 'https://api.deadlock-api.com/v1/assets/items'
response = requests.get(url, timeout=30)
response.raise_for_status()
items = response.json()

mapping = {}
for item in items:
    if item['type'] == 'upgrade' and item.get('shopable', False):
        mapping[item["id"]] = item["name"]

items_df["item_name"] = items_df["item_id"].map(mapping)

In [10]:
# merge stats and items id by game time stamps of both
result = pd.merge_asof(
    items_df.sort_values('game_time_s'),
    stats_df.sort_values('time_stamp_s'),
    left_on='game_time_s',
    right_on='time_stamp_s',
    by=['account_id', 'match_id', 'hero_id'],
    direction='nearest',)

merged = result.drop(columns=['items', 'stats'])

# produce clearn items df (silver)
purchases_clean = merged[[
    'match_id', 'account_id', 'hero_id',
    'game_time_s', 'item_name', 'item_id', 'upgrade_id', 'sold_time_s',
    'flags', 'imbued_ability_id', 'upgrade_info', 'net_worth'
]]

In [11]:
purchases_clean

,match_id,account_id,hero_id,game_time_s,item_name,item_id,upgrade_id,sold_time_s,flags,imbued_ability_id,upgrade_info,net_worth
0,102753159,171964278,69,0,NaN,103496908,0,0,0,0,65537,2055
1,102735397,1009275951,81,0,NaN,3443575800,0,0,0,0,65537,1823
2,102735397,1009275951,81,0,NaN,3443575800,0,0,0,0,65537,1823
3,102753159,171964278,69,0,NaN,103496908,0,0,0,0,65537,2055
4,102735397,1009275951,81,0,NaN,3443575800,0,0,0,0,65537,1823
...,...,...,...,...,...,...,...,...,...,...,...,...
6198097,102766667,973546530,69,5447,Vortex Web,1152158042,1,0,0,0,65537,112907
6198098,102766667,973546530,69,5447,Vortex Web,1152158042,1,0,0,0,65537,112907
6198099,102766667,973546530,69,5447,Vortex Web,1152158042,1,0,0,0,65537,112907
6198100,102766667,973546530,69,5447,Vortex Web,1152158042,1,0,0,0,65537,112907



### Drop na values CHECK IF THIS ACTUALLY DROPS HERO ABILITIES AND NOT ACTUAL PURCHASED ITEMS!!


In [12]:
purchases_clean.dropna()

,match_id,account_id,hero_id,game_time_s,item_name,item_id,upgrade_id,sold_time_s,flags,imbued_ability_id,upgrade_info,net_worth
189267,102728556,434907645,3,42,High-Velocity Rounds,3077079169,1,162,1,0,65537,1808
189281,102728556,434907645,3,42,High-Velocity Rounds,3077079169,1,162,1,0,65537,1808
189324,102728556,434907645,3,42,High-Velocity Rounds,3077079169,1,162,1,0,65537,1808
189334,102728556,434907645,3,42,High-Velocity Rounds,3077079169,1,162,1,0,65537,1808
189350,102728556,434907645,3,42,High-Velocity Rounds,3077079169,1,162,1,0,65537,1808
...,...,...,...,...,...,...,...,...,...,...,...,...
6198097,102766667,973546530,69,5447,Vortex Web,1152158042,1,0,0,0,65537,112907
6198098,102766667,973546530,69,5447,Vortex Web,1152158042,1,0,0,0,65537,112907
6198099,102766667,973546530,69,5447,Vortex Web,1152158042,1,0,0,0,65537,112907
6198100,102766667,973546530,69,5447,Vortex Web,1152158042,1,0,0,0,65537,112907


In [13]:
# import plotly.express as px
#
# purchases_agg = purchases_clean[purchases_clean["hero_id"] == 77]
#
# purchases_agg = purchases_agg.groupby("item_name").agg(
#     avg_game_time_s=("game_time_s", "mean"),
#     avg_net_worth=("net_worth", "mean"),
#     n_purchases=("net_worth", "count")
# ).reset_index()
#
#
#
# fig = px.scatter(
#     purchases_agg,
#     x="avg_game_time_s",
#     y="avg_net_worth",
#     color="item_name",
#     size="n_purchases",  # optional: bubble size = how many times purchased
#     trendline="ols",
#     trendline_scope="overall"
# )
# fig.update_layout(xaxis_title="Avg Buy Time Seconds", yaxis_title="Avg Net Worth")
# fig.show()

In [14]:
url  = 'https://api.deadlock-api.com/v1/analytics/lane-soul-curve'

params = {
    "limit": 500,
    "order_by": "start_time",
    "order_direction": "desc",
    "match_mode": "ranked",
    "include_info": "true",
    "include_more_info": "true",
    "include_objectives": "true",
    "include_mid_boss": "true",
    "include_player_info": "true",
    "include_player_final_stats": "true",
    "include_player_stats": "true",
    "include_player_items": "true",
    "include_player_death_details": "true",
    "hero_ids": "77", # Only get hero ids for apollo
    "format": "json",
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()
matches = response.json()

# print(len(matches))
# print(json.dumps(matches, indent=2)[:3000])
matches_df = pd.DataFrame(matches)
matches_df.head()

""


In [17]:
import requests
import pandas as pd

hero_ids = [1, 2, 3]
rows = []

for hid in hero_ids:
    resp = requests.get(
        "https://api.deadlock-api.com/v1/analytics/item-stats",
        params={"hero_id": hid, "min_matches": 20}  # sample-size filter
    )
    for row in resp.json():
        row["hero_id"] = hid  # add it yourself — API won't
        rows.append(row)

item_df = pd.DataFrame(rows)

In [18]:
item_df.head()

,item_id,bucket,wins,losses,matches,players,avg_buy_time_s,avg_sell_time_s,avg_buy_time_relative,avg_sell_time_relative,hero_id
0,7409189,0,291067,300471,591538,182094,437.281243,1918.632338,19.496929,77.137863,1
1,26002154,0,561,636,1197,866,1026.000835,1740.104106,44.115573,69.629599,1
2,84321454,0,83766,89151,172917,69941,874.766183,2195.093198,37.851934,82.244864,1
3,98582110,0,326,362,688,478,948.864826,1985.958333,41.041925,74.924578,1
4,112198670,0,16483,17241,33724,17594,905.140256,2112.050789,40.079303,79.997962,1
